### Requeriments for match folder `partidas`

Goal: Build an ETL process and make the data available in the appropriate layer

Read -> Transform -> Write (ETL)


1. Read all files in the folder 'partidas' from the data lake
2. Define the correct data schema
3. Include a column with the date when the file was ingested
4. Save the data in **Parquet** format in the appropriate layer

In [0]:
%run "./00_config_storage"

In [0]:
%run "../Modulo 4/Functions"

In [0]:
%run "../Modulo 4/Variables"

In [0]:
display(dbutils.fs.ls(path_bronze))

In [0]:
path_file_players = f"{path_bronze}/jogadores.json"

In [0]:
df_player = spark.read.json(path_file_players)
display(df_player)

In [0]:
from pyspark.sql.functions import col

df_player = df_player.withColumn("birth_country", col("birth.country")) \
                     .withColumn("birth_date", col("birth.date")) \
                     .withColumn("birth_place", col("birth.place")) \
                     .drop("birth")

display(df_player)

In [0]:
from pyspark.sql.functions import to_date
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    DoubleType,
)

df_player = df_player.withColumn("birth_date", to_date(col("birth_date"), "yyyy-MM-dd"))

schema_players = StructType(
    fields=[
        StructField("id", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("firstname", StringType(), True),
        StructField("lastname", StringType(), True),
        StructField("nationality", StringType(), True),
        StructField("photo", StringType(), True),
        StructField("height", StringType(), True),
        StructField("weight", StringType(), True),
        StructField("age", IntegerType(), True),
        StructField("injured", BooleanType(), True),
        StructField("birth_country", StringType(), True),
        StructField("birth_date", StringType(), True),
        StructField("birth_place", StringType(), True),
    ]
)

In [0]:
df_player.display()
df_player.printSchema()

In [0]:
df_player_date = create_column_date(df_player)
display(df_player_date)

In [0]:
df_player_date.write.mode("overwrite").parquet(f"{path_silver}/jogadores")